# ColPali Document Retriever

- https://huggingface.co/blog/manu/colpali
- https://github.com/illuin-tech/colpali?source=post_page-----d3dad7ac01f0---------------------------------------

ColPali แก้ปัญหาข้างต้นด้วยแนวคิดที่เรียบง่ายแต่ทรงพลัง: ฝังภาพของหน้าเอกสารโดยตรง!

มันทำงานสองส่วนหลักๆ คือ:

1. การทำ Index (Indexing Phase)
ป้อนภาพ: แทนที่จะประมวลผลข้อความ ColPali จะรับ ภาพ (screenshots) ของหน้าเอกสาร เข้าไปโดยตรง

- Vision LLM (PaliGemma-3B): ใช้ PaliGemma-3B ในการเข้ารหัสภาพ โดยมันจะแบ่งภาพออกเป็นส่วนย่อยๆ (patches)
- Vision Transformer (SigLIP-So400m): Patch เหล่านี้จะถูกส่งผ่าน Vision Transformer
- Language Model (Gemma 2B): จากนั้น patch embeddings จะถูกฉาย (linearly projected) และป้อนเข้าสู่ Language Model (Gemma 2B) เพื่อให้ได้ contextualized patch embeddings ที่มีคุณภาพสูงใน space ของ Language Model

  ลดมิติและจัดเก็บ: สุดท้ายจะฉายไปยังมิติที่เล็กลง (D=128) เพื่อการจัดเก็บที่มีประสิทธิภาพ และสร้าง multi-vector document representation สำหรับแต่ละภาพหน้าเอกสาร

2. การค้นหา (Querying Phase)
Embed Query: เมื่อผู้ใช้ป้อนคำค้นหา (query) โมเดลภาษาจะแปลงคำค้นหานั้นให้เป็น token embeddings

- ColBERT-style "Late Interaction" (LI): นี่คือหัวใจสำคัญ! ColPali ใช้กลไก "late interaction" แบบ ColBERT เพื่อจับคู่ query tokens กับ document patches ได้อย่างมีประสิทธิภาพ

    สำหรับแต่ละคำใน query, มันจะค้นหา document patch ที่มีการแสดงผล ColPali ที่คล้ายคลึงกันมากที่สุด
    
    จากนั้นจะ รวมคะแนน ของ patch ที่คล้ายคลึงกันมากที่สุดสำหรับทุกคำใน query เพื่อให้ได้คะแนนสุดท้ายของคู่ query-document

  - ข้อดีของ ColPali
    R1: ประสิทธิภาพการค้นหาดีเยี่ยม: ColPali สามารถดึงข้อมูลที่เกี่ยวข้องกับภาพได้ดีกว่าระบบที่เน้นข้อความอย่างเห็นได้ชัด โดยเฉพาะในเอกสารที่มีภาพประกอบซับซ้อน เช่น infographics, figures, และ tables แม้แต่เอกสารที่เน้นข้อความก็ทำได้ดีกว่า

    R2: ความเร็วในการทำ Indexing สูง: เนื่องจากไม่ต้องผ่านขั้นตอนซับซ้อนอย่าง OCR หรือการวิเคราะห์ Layout ทำให้การทำ Index เร็วขึ้นมาก

    R3: Latency ในการ Query ต่ำ: การใช้ late interaction แบบ ColBERT ช่วยให้การค้นหาทำได้รวดเร็ว

    ตีความได้ (Interpretability): ColPali สามารถแสดงให้เห็นว่า patch ส่วนไหนของเอกสารที่มีความโดดเด่นเมื่อเทียบกับคำค้นหา ซึ่งช่วยให้เข้าใจว่าโมเดล "เห็น" อะไรในภาพ

- ViDoRe Benchmark
  ColPali ยังมาพร้อมกับ ViDoRe (Visual Document Retrieval Benchmark) ซึ่งเป็น Benchmark ใหม่ที่ออกแบบมาเพื่อประเมินความสามารถของโมเดลในการค้นหาข้อมูลจากเอกสารที่ เน้นภาพ ไม่ใช่แค่ข้อความเหมือนเดิมๆ ViDoRe ครอบคลุมงานต่างๆ ในหลายหัวข้อ หลายรูปแบบ (figures, tables, text) และหลายภาษา

  สรุปง่ายๆ คือ ColPali เป็นก้าวสำคัญในการทำ document retrieval โดยเปลี่ยนจากการเน้นข้อความเป็น การเน้นภาพ โดยตรง ซึ่งช่วยลดความซับซ้อน เพิ่มประสิทธิภาพ และสามารถจัดการกับข้อมูลภาพในเอกสารได้อย่างที่ไม่เคยมีมาก่อนเลย!

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
!pip uninstall -y bitsandbytes accelerate transformers torch torchvision torchaudio

Found existing installation: accelerate 1.8.1
Uninstalling accelerate-1.8.1:
  Successfully uninstalled accelerate-1.8.1
Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
Found existing installation: torch 2.6.0+cpu
Uninstalling torch-2.6.0+cpu:
  Successfully uninstalled torch-2.6.0+cpu
Found existing installation: torchvision 0.21.0+cpu
Uninstalling torchvision-0.21.0+cpu:
  Successfully uninstalled torchvision-0.21.0+cpu
Found existing installation: torchaudio 2.6.0+cpu
Uninstalling torchaudio-2.6.0+cpu:
  Successfully uninstalled torchaudio-2.6.0+cpu


In [3]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 94.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 47.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 119.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 22.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.3/9

In [4]:
!pip install --upgrade datasets transformers
!pip install --upgrade bitsandbytes accelerate
!pip install --upgrade sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Successfully uninstalled fsspec-2025.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.7.19 requires torch<2.7,>=1.10, but you have torch 2.7.1+cu118 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 19.0 MB/s eta 0:00:00
   ━━

In [5]:
!nvcc --version

/bin/bash: line 1: nvcc: command not found


## Load Dataset

In [6]:
from datasets import load_dataset, disable_caching, enable_caching
import shutil
import os

In [7]:
# # Prevent Loading a dataset cached in a LocalFileSystem is not supported.
# disable_caching()

# cache_dir = os.path.expanduser("~/.cache/huggingface/datasets")
# if os.path.exists(cache_dir):
#   print(f"Clearing cache directory: {cache_dir}")
#   shutil.rmtree(cache_dir)
# else:
#   print(f"Cache directory not found: {cache_dir}")

# try:
#     # ลองโหลด dataset ใหม่
#     dataset = load_dataset("davanstrien/ufo-ColPali")
#     dataset = dataset["train"]
#     dataset = dataset.filter(lambda example: example["specific_detail_query"] is not None)
#     print(dataset)

# except Exception as e:
#     print(f"An error occurred: {e}")

# finally:
#     enable_caching()

In [8]:
# dataset = load_dataset("davanstrien/ufo-ColPali")
# dataset = dataset["train"]
# dataset = dataset.filter(lambda example: example["specific_detail_query"] is not None)
# dataset

In [9]:
custom_cache_dir = "./huggingface_cache_for_datasets"

# ตรวจสอบว่ามีโฟลเดอร์แคช custom_cache_dir อยู่หรือไม่ แล้วลบมัน (เพื่อความชัวร์)
if os.path.exists(custom_cache_dir):
    print(f"Clearing custom cache directory: {custom_cache_dir}")
    shutil.rmtree(custom_cache_dir)
else:
    print(f"Custom cache directory not found: {custom_cache_dir}")

# สร้างโฟลเดอร์ใหม่เผื่อไว้ (ถ้ามันยังไม่มี)
os.makedirs(custom_cache_dir, exist_ok=True)
print(f"Ensured custom cache directory exists: {custom_cache_dir}")

# ปิดการแคชชั่วคราว (เพื่อไม่ให้มันไปยุ่งกับ default cache path)
disable_caching()

try:
    # ลองโหลด dataset ใหม่ โดยระบุ cache_dir ที่เราต้องการ
    # การระบุ cache_dir ตรงๆ อาจช่วยเลี่ยงปัญหา LocalFileSystem ที่ library พยายามเข้าถึงเอง
    print("Attempting to load dataset with specified cache_dir...")
    dataset = load_dataset("davanstrien/ufo-ColPali", cache_dir=custom_cache_dir)

    dataset = dataset["train"]
    dataset = dataset.filter(lambda example: example["specific_detail_query"] is not None)
    print("Dataset loaded and filtered successfully:")
    print(dataset)

except NotImplementedError as e:
    print(f"Still encountering NotImplementedError: {e}")
    print("This might indicate an inherent limitation or a specific version issue with the 'datasets' library.")
    print("Consider updating the 'datasets' library or checking its GitHub issues for similar reports.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

finally:
    # เปิดการแคชกลับคืน (สำคัญ!)
    enable_caching()


Custom cache directory not found: ./huggingface_cache_for_datasets
Ensured custom cache directory exists: ./huggingface_cache_for_datasets
Attempting to load dataset with specified cache_dir...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/293M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2243 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2243 [00:00<?, ? examples/s]

Dataset loaded and filtered successfully:
Dataset({
    features: ['image', 'raw_queries', 'broad_topical_query', 'broad_topical_explanation', 'specific_detail_query', 'specific_detail_explanation', 'visual_element_query', 'visual_element_explanation', 'parsed_into_json'],
    num_rows: 2172
})


## Copali Model

In [10]:
import torch
from transformers import ColPaliForRetrieval, ColPaliProcessor, BitsAndBytesConfig

Copali Model https://huggingface.co/vidore/colpali-v1.2-hf



```
import torch
from PIL import Image

from transformers import ColPaliForRetrieval, ColPaliProcessor

model_name = "vidore/colpali-v1.2-hf"

model = ColPaliForRetrieval.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",  # or "mps" if on Apple Silicon
).eval()

processor = ColPaliProcessor.from_pretrained(model_name)

# Your inputs
images = [
    Image.new("RGB", (32, 32), color="white"),
    Image.new("RGB", (16, 16), color="black"),
]
queries = [
    "What is the organizational structure for our R&D department?",
    "Can you provide a breakdown of last year’s financial performance?",
]

# Process the inputs
batch_images = processor(images=images).to(model.device)
batch_queries = processor(text=queries).to(model.device)

# Forward pass
with torch.no_grad():
    image_embeddings = model(**batch_images)
    query_embeddings = model(**batch_queries)

# Score the queries against the images
scores = processor.score_retrieval(query_embeddings.embeddings, image_embeddings.embeddings)

```



In [11]:
model_name = "vidore/colpali-v1.2-hf"

# # Define Model and Processor -> Vision Language Model
# model_vlm =  ColPaliForRetrieval.from_pretrained(model_name,
#                                                  torch_dtype=torch.bfloat16,
#                                                  device_map="cuda" if torch.cuda.is_available() else "mps").eval()

# model_processor = ColPaliProcessor.from_pretrained(model_name)

To Reduce GPU RAM, we will try to used bits and bytes for Quantization.

In [12]:
import bitsandbytes as bnb

In [13]:
# Define Model and Processor -> Vision Language Model
# quantization_config_8bit = BitsAndBytesConfig(
#     load_in_8bit=True
# )

model_name = "vidore/colpali-v1.2-hf"
model_vlm =  ColPaliForRetrieval.from_pretrained(model_name,
                                                #  quantization_config=quantization_config_8bit,
                                                 torch_dtype=torch.bfloat16,
                                                 device_map="auto"
)

model_vlm = model_vlm.eval()
model_processor = ColPaliProcessor.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/862M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

ImportError: /usr/local/lib/python3.11/dist-packages/_XLAC.cpython-311-x86_64-linux-gnu.so: undefined symbol: _ZN5torch4lazy13MetricFnValueEd

## Test Embedding Text to Vector

In [ ]:
inputs = model_processor(text="a document about Mars expedition").to('cuda')
"""
{'input_ids': tensor([[2, 9413, 235292, 476, 4551, 1105, 16359, 40753, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 108]],
  device='cuda:0'),
  'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
  device='cuda:0')
}
"""

inputs

In [ ]:
with torch.no_grad():
  text_embeds = model_vlm(**inputs, return_tensors="pt").embeddings

text_embeds
"""
tensor([[[ 0.0249,  0.1211,  0.0364,  ...,  0.0830, -0.0359,  0.0405],
         [ 0.0099, -0.0121,  0.0025,  ...,  0.0173,  0.0410, -0.1562],
         [ 0.0620, -0.0222, -0.1895,  ..., -0.0967,  0.0347,  0.0220],
         ...,
         [ 0.0530, -0.0537,  0.1143,  ...,  0.0913, -0.0251,  0.0596],
         [ 0.0457, -0.0265,  0.1084,  ...,  0.0757, -0.0579,  0.0693],
         [ 0.2168,  0.1523, -0.0525,  ...,  0.1650,  0.0383, -0.0469]]],
       device='cuda:0', dtype=torch.bfloat16)
"""

In [ ]:
dataset

## Map Dataset to Embedding

In [ ]:
ds_with_embeddings = dataset.map(
    lambda example: {
        'embeddings': model_vlm(
            **model_processor(images=example["image"]).to("cuda"),
            return_tensors="pt"
        ).embeddings.to(torch.float32).detach().cpu().numpy()
    }
)

ds_with_embeddings

## Function to find indexes for similarity searching

In [ ]:
def find_top_k_indices_batched(dataset, text_embedding, processor, k=10, batch_size=4):
    scores_and_indices = []

    for start_idx in range(0, len(dataset), batch_size):

        end_idx = min(start_idx + batch_size, len(dataset))
        batch = dataset[start_idx:end_idx]
        batch_embeddings = [torch.tensor(emb[0], dtype=torch.float32) for emb in batch["embeddings"]]
        scores = processor.score_retrieval(text_embedding.to("cpu").to(torch.float32), batch_embeddings)

        if hasattr(scores, "tolist"):
            scores = scores.tolist()[0]

        for i, score in enumerate(scores):
            scores_and_indices.append((score, start_idx + i))

    sorted_results = sorted(scores_and_indices, key=lambda x: -x[0])

    topk = sorted_results[:k]
    indices = [idx for _, idx in topk]
    scores = [score for score, _ in topk]

    return indices, scores

In [ ]:
with torch.no_grad():
  text_embeds = model_vlm(
      *model_processor(text="a document about Mars expedition").to("cuda"),
      return_tensors="pt"
  )

  text_embeds = text_embeds.embeddings

In [ ]:
indices, scores = find_top_k_indices_batched(
    ds_with_embeddings,
    text_embeds,
    model_processor,
    k=3,
    batch_size=4
)

print(indices, scores)

In [ ]:
for i in indices:
  display(dataset[i]["image"])

In [ ]:
with torch.no_grad():
  text_embeds = model_vlm(
      *model_processor(text="a document about Mars expedition").to("cuda"),
      return_tensors="pt"
  )

  text_embeds = text_embeds.embeddings


indices, scores = find_top_k_indices_batched(
    ds_with_embeddings,
    text_embeds,
    model_processor,
    k=3,
    batch_size=4
)

print(indices, scores)

In [ ]:
for i in indices:
  display(dataset[i]["image"])